# GraviNum - N-dimensional arrays and linear algebra

Build the solution first so the notebook can reference the assemblies:

```
dotnet build Gravicode.Science.sln -c Release
```

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviNum;
using Gravicode.Science.GraviNum.Compute;

Console.WriteLine(GraviInfo.HardwareReport());

## Creating arrays

Every array is a shared buffer plus a shape, strides and an offset, so reshaping and transposing cost nothing.

In [ ]:
var a = NdArray.Arange(12).Reshape(3, 4);
Console.WriteLine(a);

Console.WriteLine($"transpose shares the buffer: a[1,2]={a[1, 2]}, a.T[2,1]={a.T[2, 1]}");
Console.WriteLine($"a[:, 1:3] is a strided view, contiguous = {a.Slice(Slice.All, Slice.Range(1, 3)).IsContiguous}");

## Broadcasting

A length-4 vector stretches across all three rows without being copied.

In [ ]:
var matrix = NdArray.Ones(3, 4);
var offsets = NdArray.Arange(4);
Console.WriteLine(matrix + offsets);

## Linear algebra

In [ ]:
var m = NdArray.FromArray(new double[,] { { 4, 7, 2 }, { 3, 6, 1 }, { 2, 5, 9 } });

Console.WriteLine($"det  = {LinAlg.Determinant(m):F4}");
Console.WriteLine($"rank = {LinAlg.MatrixRank(m)}");
Console.WriteLine($"cond = {LinAlg.ConditionNumber(m):F4}");

var svd = Decomposition.Svd(m);
Console.WriteLine($"singular values: {string.Join(", ", svd.SingularValues.ToArray().Select(v => v.ToString("F4")))}");
Console.WriteLine($"U S V' reconstructs A: {UFunc.AllClose(svd.Reconstruct(), m, 1e-9)}");

## Statistics

In [ ]:
var rng = new GraviRandom(42);
var samples = rng.Normal(100, 15, 100_000);

foreach (var (key, value) in Statistics.Describe(samples))
    Console.WriteLine($"{key,-8}{value,12:F4}");

## Heatmap

ScottPlot renders directly into the notebook.

In [ ]:
var heat = NdArray.Zeros(40, 40);
for (var i = 0; i < 40; i++)
    for (var j = 0; j < 40; j++)
    {
        var u = (i - 20) / 8.0;
        var v = (j - 20) / 8.0;
        heat[i, j] = Math.Exp(-(u * u + v * v) / 2) * Math.Cos(u * 2) * Math.Sin(v * 2);
    }

var plot = new ScottPlot.Plot();
plot.Add.Heatmap(heat.To2DArray());
plot.Title("GraviNum - matrix heatmap");
plot.GetPngHtml(800, 600)

## CPU versus GPU

The GPU only wins once the matrix is big enough to hide the transfer cost.

In [ ]:
using System.Diagnostics;

foreach (var size in new[] { 128, 256, 512, 1024 })
{
    var left = rng.StandardNormal(size, size);
    var right = rng.StandardNormal(size, size);

    var watch = Stopwatch.StartNew();
    LinAlg.Dot(left, right);
    watch.Stop();

    var gflops = 2.0 * size * size * size / watch.Elapsed.TotalSeconds / 1e9;
    Console.WriteLine($"{size,5} x {size,-5}{watch.ElapsedMilliseconds,7} ms{gflops,9:F2} GFLOP/s");
}

Console.WriteLine(Compute.DescribeDevices());

## Einstein summation

One notation for transposing, tracing, contracting and outer products. Every letter that appears in
the inputs but not the output is summed over — that single rule covers most of tensor algebra.

The value is that the intent is written down: `LinAlg.Dot(a.T, b)` makes you reconstruct which axes
met, and `"ji,jk->ik"` says so.


In [ ]:
var ea = rng.StandardNormal(4, 5);
var eb = rng.StandardNormal(5, 3);

Console.WriteLine($"\"ij,jk->ik\" == LinAlg.Dot : {UFunc.AllClose(Einsum.Evaluate("ij,jk->ik", ea, eb), LinAlg.Dot(ea, eb), 1e-12)}");

var square = NdArray.FromArray(new double[,] { { 1, 2, 3 }, { 4, 5, 6 }, { 7, 8, 9 } });
Console.WriteLine($"\"ii->i\"  diagonal : [{string.Join(", ", Einsum.Evaluate("ii->i", square).ToArray())}]");
Console.WriteLine($"\"ii->\"   trace    : {Einsum.Evaluate("ii->", square).At(0)}");
Console.WriteLine($"\"ij->j\"  col sums : [{string.Join(", ", Einsum.Evaluate("ij->j", square).ToArray())}]");

// Rank 3 is where writing this out by hand stops being readable.
var batched = Einsum.Evaluate("bij,bjk->bik", rng.StandardNormal(3, 4, 5), rng.StandardNormal(3, 5, 2));
Console.WriteLine($"\"bij,bjk->bik\"    : shape [{string.Join(", ", batched.Shape.ToArray())}]");


## Complex arrays

`ComplexNdArray` stores interleaved real/imaginary pairs — the same layout as FFTW and NumPy — so
the buffer goes straight to the FFT without a repack.

`ConjugateTranspose` is what nearly every complex formula means. `AᴴA` is positive semi-definite
with a real diagonal; `AᵀA` is neither, and gives complex "variances" that pass every shape check.


In [ ]:
using System.Numerics;
using Gravicode.Science.GraviNum.Signal;

var z = ComplexNdArray.FromParts(rng.StandardNormal(3, 3), rng.StandardNormal(3, 3));

var gram = ComplexNdArray.Dot(z.ConjugateTranspose(), z);
Console.WriteLine("A^H A diagonal (real, non-negative):");
for (var i = 0; i < 3; i++)
    Console.WriteLine($"  [{i},{i}] = {gram[i, i].Real:F4} + {gram[i, i].Imaginary:E1}i");

var signal = ComplexNdArray.FromParts(rng.StandardNormal(64), rng.StandardNormal(64));
Console.WriteLine($"\nFft round trip holds : {signal.Fft().Ifft().AllClose(signal, 1e-10)}");

// Parseval: energy is conserved, which pins the scaling convention.
var timeEnergy = 0.0;
for (var i = 0; i < signal.Size; i++) timeEnergy += signal.Power().At(i);
var spectrum = signal.Fft();
var freqEnergy = 0.0;
for (var i = 0; i < spectrum.Size; i++) freqEnergy += spectrum.Power().At(i);
Console.WriteLine($"Parseval: {timeEnergy:F4} vs {freqEnergy / signal.Size:F4}");


## Power spectrum

Two tones buried in noise. The transform recovers both peaks and leaves everything else as a floor —
which is the whole reason to look at a signal in the frequency domain.


In [ ]:
const int n = 512;
const double sampleRate = 256.0;

var tone = NdArray.Zeros(n);
for (var i = 0; i < n; i++)
{
    var t = i / sampleRate;
    tone.SetAt(i, Math.Sin(2 * Math.PI * 12 * t) + 0.5 * Math.Sin(2 * Math.PI * 40 * t) + 0.15 * rng.Normal());
}

var bins = Fft.FrequencyBins(n, sampleRate).ToArray().Take(n / 2).ToArray();
var power = Fft.Magnitude(tone.ToArray()).ToArray().Take(n / 2).ToArray();

var spectrumPlot = new ScottPlot.Plot();
spectrumPlot.Add.Scatter(bins, power);
spectrumPlot.Title("Power spectrum: 12 Hz and 40 Hz recovered from noise");
spectrumPlot.XLabel("frequency (Hz)");
spectrumPlot.YLabel("magnitude");
spectrumPlot.GetPngHtml(800, 400)


## Slice ergonomics

`Slice` returns a **view**, so writing through it changes the original. Anything that returns an
array returns a copy; anything that writes, writes through.

`SliceEllipsis` names a trailing axis without counting the leading ones, so the same expression keeps
working if the rank grows.


In [ ]:
var grid = NdArray.Arange(12).Reshape(3, 4).Copy();
Console.WriteLine("before:");
Console.WriteLine(grid);

grid.Slice(Slice.Range(1, 2)).Assign(0.0);
Console.WriteLine("after grid[1:2] = 0 — the write went through the view:");
Console.WriteLine(grid);

var volume = NdArray.Arange(24).Reshape(2, 3, 4);
Console.WriteLine($"SliceEllipsis([], [Slice.At(3)]) -> shape [{string.Join(", ", volume.SliceEllipsis([], [Slice.At(3)]).Shape.ToArray())}]");

var table = NdArray.Arange(12).Reshape(3, 4);
Console.WriteLine($"TakeAlong([3, 0], axis: 1)       -> shape [{string.Join(", ", table.TakeAlong([3, 0], axis: 1).Shape.ToArray())}]");
Console.WriteLine($"IndicesWhere(v => v > 8)         -> [{string.Join(", ", table.IndicesWhere(v => v > 8))}]");
